# 06 · Model depth slices as GeoTIFFs

Writes depth slices of a model as GeoTIFF rasters, to use in QGIS or ArcGIS with your other maps.
Each pixel gets the value of the model cell under the pixel's centre.

* **Needs:** the model `.vtu`, its receivers, and its frame (as in notebook 05).
* **Writes:** `work/06_geotiff/depth_slice_<depth>m.tif`, one per depth, or one multi-band file.
* **Takes:** seconds per slice at 1 km pixels; smaller pixels take longer.
* **GUI:** plotting GUI, *GeoTIFF* tab.

## Settings

In [ ]:
MODEL_VTU = "work/04_refine/synthetic/model_synthetic.vtu"
RECEIVERS = "work/04_refine/synthetic/receivers_projected.csv"
MAKE_SAMPLE_MODEL = True       # sample only: make the stand-in model if notebook 04 hasn't

FRAME_FILE = "work/02_survey/model_frame.json"   # or None to fit the frame from EDI_DIR
EDI_DIR = "sample_data/edis"
DISPLAY_CRS = "EPSG:32610"     # the rasters' projection; "" for the model's own frame

DEPTHS_M = [0, 1000, 2500, 5000, 10000, 20000, 40000]   # metres below the REFERENCE
REFERENCE = "sea_level"        # "sea_level" or "surface" (depth below the model's surface)
PIXEL_M = 1000.0               # pixel size (m)
PADDING_M = 20000.0            # how far past the stations the rasters reach (m)
QUANTITY = "log10_rho"         # "log10_rho", "rho", "log10_sigma" or "sigma"
LAYOUT = "per_depth"           # "per_depth": one file per depth; "stack": one file, one band per depth
PREFIX = "depth_slice"
OUT_DIR = "work/06_geotiff"

In [ ]:
%matplotlib inline
import os

import matplotlib.pyplot as plt
import numpy as np
import rasterio

from mtinv_gui.backends import projection as P
from mtinv_gui.plotting import project as PP
from mtinv_gui.plotting.backends import geotiff
from mtinv_gui.plotting.backends import georef as G
from mtinv_gui.plotting.backends.session import PlotSession

## 1. Load the model and put it on the map

The same as notebook 05.

In [ ]:
if MAKE_SAMPLE_MODEL and not os.path.isfile(MODEL_VTU):
    import sys
    sys.path.insert(0, "sample_data")
    from synthetic_model import make_synthetic_inversion
    make_synthetic_inversion("work/03_mesh", os.path.dirname(MODEL_VTU))

session = PlotSession()
session.load_model(PP.ModelInfo(model_path=MODEL_VTU, receivers_path=RECEIVERS))
if FRAME_FILE:
    frame_georef = G.from_frame(P.load_frame(FRAME_FILE))
    georef = PP.GeorefInfo(method="manual", manual_crs=frame_georef.crs, manual_x0=frame_georef.x0,
                           manual_y0=frame_georef.y0, display_crs=DISPLAY_CRS)
else:
    georef = PP.GeorefInfo(method="stations", station_source=EDI_DIR, display_crs=DISPLAY_CRS)
session.solve_georef(georef)
print(f"{session.model.n_earth_cells:,} earth cells; rasters in {session.maptr.display_label}")

## 2. The raster grid

The extent is the stations plus `PADDING_M`, snapped to whole pixels so rasters from different
runs line up.

In [ ]:
x_min, x_max, y_min, y_max = session.extent(PP.ExtentInfo(mode="stations", padding_m=PADDING_M))
grid = geotiff.RasterGrid.from_bounds(x_min, x_max, y_min, y_max, PIXEL_M)
print(f"{grid.nx} x {grid.ny} pixels of {PIXEL_M:g} m, {len(DEPTHS_M)} depths")

## 3. Write

In [ ]:
paths = geotiff.export(session.model, session.maptr, DEPTHS_M, grid, OUT_DIR, reference=REFERENCE,
                       quantity=QUANTITY, layout=LAYOUT, prefix=PREFIX)
for path in paths:
    print(path)

## 4. Check

Read the files back and show them. Pixels outside the model or above the surface are nodata
(white).

In [ ]:
bands = []
for path in paths:
    with rasterio.open(path) as src:
        for b in range(1, src.count + 1):
            bands.append((src.descriptions[b - 1] or os.path.basename(path), src.read(b, masked=True), src.bounds))

ncol = min(4, len(bands))
nrow = int(np.ceil(len(bands) / ncol))
scale = {"log10_rho": dict(vmin=0, vmax=4, cmap="turbo_r"), "log10_sigma": dict(vmin=-4, vmax=0, cmap="turbo")}
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.6 * nrow), squeeze=False)
for ax, (name, values, b) in zip(axes.ravel(), bands):
    im = ax.imshow(values, extent=(b.left, b.right, b.bottom, b.top), **scale.get(QUANTITY, {}))
    ax.set_title(name, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes.ravel()[len(bands):]:
    ax.set_axis_off()
fig.colorbar(im, ax=axes, shrink=0.6, label=QUANTITY)
plt.show()

**For coders:** `geotiff.sample_slice(model, maptr, grid, depth_m)` returns one slice as an array,
to compute your own rasters (differences between models, averages over a depth range, ...) before
writing them with rasterio.